In [ ]:
# Import path to source directory (bit of a hack in Jupyter)
import sys
import os
pwd = %pwd
sys.path.append(os.path.join(pwd, '..'))
sys.path.append(os.path.join(pwd, 'src'))

# Ensure modules are reloaded on any change (very useful when developing code on the fly)
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import json
import os
from programmable_cubes_UDP import programmable_cubes_UDP
PREFIX = "../"

# Creation of additional problems

In [ ]:
def create_problem(name, data, initial_config, target_config, initial_types, target_types):
    global PREFIX
    json_data = json.dumps(data,indent=4)
    try:
        os.mkdir(f"{PREFIX}data/{name}")
    except:
        print("directory already exists")
    with open(f"{PREFIX}problems/{name}.json","w") as f:
        f.write(json_data)
    np.save(f'{PREFIX}data/{name}/Target_Config.npy',target_config)
    np.save(f'{PREFIX}data/{name}/Initial_Config.npy',initial_config)
    np.save(f'{PREFIX}data/{name}/Target_Cube_Types.npy',target_types)
    np.save(f'{PREFIX}data/{name}/Initial_Cube_Types.npy',initial_types)

## 1) test - 3 cubes with 1 mistake

In [ ]:
PROBLEM_NAME = "test_2"

initial_config = np.array([[0,0,0],[1,0,0],[-1,0,0]])
target_config  = np.array([[0,0,0],[0,1,0],[-1,0,0]])
initial_types = np.array([0,0,1])
target_types = np.array([0,0,1])
data = {
 "path": f"data/{PROBLEM_NAME}",
 "plot_dim": 5,
 "max_cmds": 100, 
 "num_cubes": 3,
 "num_cube_types": 2,
 "fitness_offset": 0.66,
 "colours": ["lightgray",
            "limegreen"]
}


create_problem(PROBLEM_NAME,data,initial_config,target_config,initial_types,target_types)

## 2) WALL - large wall with 1 mistake

In [ ]:
PROBLEM_NAME = "WALL"

initial_config = np.array([[i,j,0] for i in np.arange(0,10) for j in np.arange(0,10)])
target_config  = initial_config
initial_config = np.concat([initial_config,np.array([[-1,0,0]])])
target_config = np.concat([target_config  ,np.array([[0,10,0]])])
initial_types = np.array([0 for i in np.arange(0,len(initial_config))])
target_types = initial_types
data = {
 "path": f"data/{PROBLEM_NAME}",
 "plot_dim": 15,
 "max_cmds": 1000, 
 "num_cubes": 101,
 "num_cube_types": 1,
 "fitness_offset": 0.0,
 "colours": ["lightgray"]
}

create_problem(PROBLEM_NAME,data,initial_config,target_config,initial_types,target_types)


## 3) WALL_MORE_ERRORS - large wall with 3 mistakes

In [ ]:
PROBLEM_NAME = "WALL_MORE_ERRORS"

initial_config = np.array([[i,j,0] for i in np.arange(0,10) for j in np.arange(0,10)])
target_config  = initial_config
initial_config = np.concat([initial_config,np.array([[-1,0,0],[0,-1,0],[10,9,0]])])
target_config = np.concat([target_config  ,np.array([[0,10,0],[10,5,0],[5,5,1]])])
initial_types = np.array([0 for i in np.arange(0,len(initial_config))])
target_types = initial_types
data = {
 "path": f"data/{PROBLEM_NAME}",
 "plot_dim": 15,
 "max_cmds": 1000, 
 "num_cubes": 101,
 "num_cube_types": 1,
 "fitness_offset": 0.0,
 "colours": ["lightgray"]
}

create_problem(PROBLEM_NAME,data,initial_config,target_config,initial_types,target_types)


## 4) BRIDGE_FIX - cube with wrong type stuck between two cubes 

In [ ]:
PROBLEM_NAME = "BRIDGE_FIX"

initial_config = np.array([[i,0,0] for i in np.arange(10)])
initial_config = np.concatenate([initial_config,np.array([[0,-1,0],[0,1,0],[0,0,1],[0,0,-1],[-1,0,0]])])
target_config  = initial_config
initial_types = np.array([0 for i in np.arange(0,len(initial_config))])
target_types = np.array(initial_types)
initial_types[4] = 1
target_types[len(target_types)-1] = 1
data = {
 "path": f"data/{PROBLEM_NAME}",
 "plot_dim": 15,
 "max_cmds": 10000, 
 "num_cubes": len(initial_config),
 "num_cube_types": 2,
 "fitness_offset": 0.0,
 "colours": ["lightgray","blue"]
}
create_problem(PROBLEM_NAME,data,initial_config,target_config,initial_types,target_types)

## 5) Inverse problems

In [ ]:
# Load original problem
PROBLEM_NAME = "JWST"
udp = programmable_cubes_UDP(PROBLEM_NAME,PREFIX)
udp.fitness(np.array([-1]))

# Invert and save with INV suffix
ti = np.array(udp.initial_cube_types)
ci = np.array(udp.final_cube_positions)
ct = np.array(udp.target_cube_positions)
tt = np.array(udp.target_cube_types)
types = np.arange(np.max(ti)+1)

INVERTED_PROBLEM_NAME = PROBLEM_NAME + "_INV"

json_data = ""
with open(f"{PREFIX}problems/{PROBLEM_NAME}.json","r") as f:
    json_data += f.read()
data = json.JSONDecoder().decode(json_data)
data["path"] = f"data/{INVERTED_PROBLEM_NAME}"

create_problem(INVERTED_PROBLEM_NAME,data,ci,ct,ti,tt)
